# Exercise: Multi-Tier Retrieval System
## Building a Resilient Context Retrieval Pipeline

### Objective

Build a **multi-tier retrieval system** that combines:
1. **Caching**: Fast retrieval for repeated queries
2. **Vector Search**: Semantic matching for new queries
3. **Fallback Search**: Simple keyword search if vector search fails

This exercise teaches resilience and efficiency in retrieval systems.

### Why Multi-Tier Retrieval?

Real-world systems need:
- **Performance**: Cache speeds up repeated queries (99% of production traffic)
- **Flexibility**: Vector search handles semantic queries
- **Resilience**: Fallback ensures results even if vector DB fails
- **Cost Efficiency**: Each tier has different speed/cost tradeoffs

### Architecture

```
User Query
    ↓
[Tier 1: Cache] ← Fast, but limited
    ↓ (miss)
[Tier 2: Vector DB] ← Semantic, powerful
    ↓ (miss)
[Tier 3: Fallback] ← Simple, always works
    ↓
Results → Cache for future
```

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# -------------------------
# Sample Documents
# -------------------------
documents = [
    Document(page_content="Revenue increased by 30% in Q4", metadata={"topic": "finance"}),
    Document(page_content="Customers prefer subscription pricing", metadata={"topic": "product"}),
    Document(page_content="AI adoption is growing in enterprises", metadata={"topic": "market"})
]

# -------------------------
# Layer 1: Cache (In-Memory)
# -------------------------
cache = {}   # TODO: key = query, value = results


# -------------------------
# Layer 2: Vector Store
# -------------------------
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma.from_documents(
    documents,
    embedding=embeddings,
    collection_name="multi_tier_demo"
)


# -------------------------
# Layer 3: Raw Fallback
# -------------------------
def fallback_search(query):
    # TODO: simple keyword search
    results = []
    for doc in documents:
        if query.lower() in doc.page_content.lower():
            results.append(doc)
    return results


# -------------------------
# Multi-Tier Retrieval
# -------------------------
def retrieve_context(query):

    # TODO 1: Check cache
    if query in cache:
        print("Cache HIT")
        return cache[query]

    print("Cache MISS → Checking Vector DB")

    # TODO 2: Vector search
    results = vector_store.similarity_search(query, k=2)

    # TODO 3: If no results → fallback
    if not results:
        print("Vector MISS → Falling back")
        results = fallback_search(query)

    # TODO 4: Store in cache
    cache[query] = results

    return results


# -------------------------
# Test Queries
# -------------------------
queries = [
    "pricing strategy",
    "enterprise growth",
    "pricing strategy"   # should hit cache
]

for q in queries:
    print(f"\nQuery: {q}")
    docs = retrieve_context(q)

    for d in docs:
        print("-", d.page_content)

## Instructions

### Part 1: Setup (Already Done ✓)

The following are already configured:
- Sample documents with metadata
- Embeddings model (text-embedding-3-small)
- Vector store (Chroma) with documents
- Fallback search function (simple keyword matching)

### Part 2: Implement Multi-Tier Retrieval

Your task is to complete the `retrieve_context()` function with the following logic:

#### TODO 1: Check Cache
- **Line**: Where it says `# TODO 1: Check cache`
- **Task**: If the query is already in the `cache` dictionary, return the cached results immediately
- **Hint**: Use a simple `if query in cache:` check
- **Expected Output**: Should print "Cache HIT" when retrieving the same query twice

#### TODO 2: Vector Search
- **Line**: Where it says `# TODO 2: Vector search`
- **Task**: Use the vector store's `similarity_search()` method with k=2
- **Hint**: `vector_store.similarity_search(query, k=2)`
- **Expected Output**: Should return semantically similar documents

#### TODO 3: Fallback Search
- **Line**: Where it says `# TODO 3: If no results → fallback`
- **Task**: If vector search returns no results, use the fallback_search function
- **Hint**: Check `if not results:`
- **Expected Output**: Should print "Vector MISS → Falling back" and return keyword matches

#### TODO 4: Cache the Results
- **Line**: Where it says `# TODO 4: Store in cache`
- **Task**: Store the results in the cache dictionary for future queries
- **Hint**: `cache[query] = results`
- **Expected Output**: Next identical query should hit the cache

## Testing Your Implementation

The test section at the end runs three queries:

```python
queries = [
    "pricing strategy",        # First query - tests vector search
    "enterprise growth",       # Second query - different topic
    "pricing strategy"         # Repeat - should hit cache
]
```

### Expected Output

When you run the code successfully, you should see:

```
Query: pricing strategy
Cache MISS → Checking Vector DB
- [document content from vector search]

Query: enterprise growth
Cache MISS → Checking Vector DB
- [document content from vector search]

Query: pricing strategy
Cache HIT
- [same document as first query]
```

**Key Observations:**
1. First "pricing strategy" → Cache MISS, Vector Search
2. Second "enterprise growth" → Different query, Cache MISS, Vector Search
3. Third "pricing strategy" → Exact match, **Cache HIT** (no vector search needed!)

### Evaluation Criteria

Your implementation is correct if:
- ✓ Cache hits are detected and printed
- ✓ Cache misses trigger vector search
- ✓ Repeated queries return instantly from cache
- ✓ No errors during execution
- ✓ Results are relevant to queries

## Challenges & Extensions (Optional)

Once you complete the basic exercise, try these enhancements:

### Challenge 1: Cache Expiration
Add a timestamp to cache entries and invalidate them after 5 minutes:
```python
import time
cache = {}  # Now: {query: (results, timestamp)}
```

### Challenge 2: Hit Rate Monitoring
Track how many cache hits vs. misses you get:
```python
cache_hits = 0
cache_misses = 0
```

### Challenge 3: Tier-Specific Metrics
Print performance metrics:
- Time for cache lookup
- Time for vector search
- Fallback frequency

### Challenge 4: Weighted Results
Modify fallback search to return a weighted score based on relevance:
```python
# Results from cache/vector: confidence=1.0
# Results from fallback: confidence=0.5
```

### Challenge 5: Query Preprocessing
Normalize queries before caching (lowercase, remove punctuation):
```python
def normalize_query(q):
    return q.lower().strip().replace("?", "")
```

---

## Summary

You've built a **resilient, efficient retrieval system** with:
- **Caching** for instant repeated queries
- **Vector search** for semantic understanding
- **Fallback** for robustness

This pattern is used in production systems at scale!